# 5. Trees and parameter selection

An FMM plan is organised by an octree. This tutorial inspects the complete
**uniform tree**, shows how depth and expansion order trade near-field work
against far-field work, runs the two empirical **parameter advisers**, and
introduces the geometry-only **adaptive tree** that can feed the same static
plan.

| Concept | Where it lives |
|---|---|
| `UniformTree` | complete octree: every box at every level exists, Morton-sorted sources and targets |
| `list1` | the touching same-level neighbourhood of a box (27 boxes inside the domain): exact P2P |
| `list2` | children of the parent's neighbours that are not in `list1`: M2L |
| `UniformFmmOptions.tree.max_level` | the depth; leaves are the deepest level |
| `AdaptiveTree` | geometry-only octree that stops splitting at a leaf capacity and produces a compact topology |
| `suggest_depth_for_performance`, `suggest_parameters_for_accuracy` | advisers that measure candidate settings on your data |

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import cdfmm
from adaptive_showcase import generate_random_grains, summarise
from tutorial_utils import (
    draw_box_3d, field_error_summary, finish_3d_axes, new_3d_figure,
    nodes_at_level, print_table, quiet_construction, vec3_to_array,
)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True})

## The uniform tree

The root cube is the bounding cube of all sources and targets unless it is
given explicitly. At level $\ell$ every axis has $2^\ell$ boxes; nodes are
stored level by level in Morton order, and sources and targets are sorted
independently by the Morton index of their leaf, so each leaf owns a
contiguous range of the sorted arrays.

In [ ]:
rng = np.random.default_rng(42)
sources = rng.normal(size=(150, 3)) * np.array([1.4, 1.0, 0.7])

tree_options = cdfmm.UniformTreeOptions()
tree_options.max_level = 2
tree = cdfmm.UniformTree(sources, tree_options)
leaves = nodes_at_level(tree, tree.leaf_level)

print(f"root centre {vec3_to_array(tree.root_centre)}, half-width {tree.root_half_width:.3f}")
print(f"levels {tree.n_levels}, nodes {len(tree.nodes)}, leaves {len(leaves)}, "
      f"occupied leaves {sum(node.source_count > 0 for node in leaves)}")
print("source permutation (first 10):", tree.source_permutation[:10])

selected = next(node for node in leaves if (node.ix, node.iy, node.iz) == (1, 1, 1))
figure, axes = new_3d_figure(figsize=(8, 7))
for node in leaves:
    draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="0.8",
                linewidth=0.4, alpha=0.4)
for slot, index in enumerate(selected.list2):
    node = tree.nodes[index]
    draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="tab:orange",
                linewidth=1.1, alpha=0.9, label="list2: M2L" if slot == 0 else None)
for slot, index in enumerate(selected.list1):
    node = tree.nodes[index]
    draw_box_3d(axes, vec3_to_array(node.centre), node.half_width, colour="tab:blue",
                linewidth=1.4, alpha=0.95, label="list1: exact P2P" if slot == 0 else None)
draw_box_3d(axes, vec3_to_array(selected.centre), selected.half_width, colour="tab:red",
            linewidth=2.8, label="selected leaf")
axes.scatter(*sources.T, s=8, color="tab:blue", alpha=0.5)
finish_3d_axes(axes, f"Interaction lists of leaf {selected.index}: "
                     f"{len(selected.list1)} near, {len(selected.list2)} far boxes")
axes.legend(loc="upper left", fontsize=8)
figure.tight_layout()

## Depth moves work between the near and the far field

The exact near field costs one pair per source in a leaf's `list1`, the far
field one M2L translation per `list2` entry. Deeper trees have smaller leaves:
fewer near-field pairs, more translations and more (mostly empty) boxes. The
topology counts below come from `uniform_topology`, without building any
operator.

One geometric caveat applies to regular lattices: if the lattice spacing
divides the root box, every body sits exactly on a box boundary, and a source
at a box corner is the worst case for the expansion (its distance from the
centre is the half-diagonal). Giving the tree an explicit root that is offset
from the lattice planes restores the ordinary convergence; the adaptive
comparison below does exactly that.

In [ ]:
rows = []
for depth in (1, 2, 3, 4):
    options = cdfmm.UniformTreeOptions()
    options.max_level = depth
    topology = cdfmm.uniform_topology(cdfmm.UniformTree(sources, sources, options))
    nodes = topology.nodes
    pairs = sum(nodes[record.source_leaf].source_count * nodes[record.target_leaf].target_count
                for record in topology.p2p_leaf_records)
    occupancy = [leaf.count for leaf in topology.source_leaves]
    rows.append({
        "depth": depth, "nodes": len(nodes), "occupied leaves": len(occupancy),
        "mean occupancy": np.mean(occupancy), "near-field pairs": pairs,
        "M2L translations": len(topology.m2l_interactions),
    })
print_table(rows, formats={"mean occupancy": ".1f"})

## Advisers: measure instead of guess

Both advisers construct ordinary plans for every candidate on **your**
geometry and moments and time or sample them; they change no default and
their output is an empirical suggestion, not a bound. Copy the returned order
and depth into the production options once, at setup.

`suggest_depth_for_performance` reports near-field, far-field and total wall
time per candidate depth and picks the fastest. For a sequential backend the
total decides; `CUDA_PARTIAL` overlaps its two branches and additionally
reports `max(near, far)`.

In [ ]:
N = 3000
positions = rng.uniform(-1.0, 1.0, size=(N, 3))
targets = rng.uniform(-1.0, 1.0, size=(N, 3))
moments = rng.normal(size=(N, 3))

with quiet_construction():
    performance = cdfmm.suggest_depth_for_performance(
        positions, targets, moments, order=6,
        backend=cdfmm.ExecutionBackend.CPU_STATIC,
        candidate_depths=[1, 2, 3], repetitions=2,
    )
print(f"suggested depth for order 6: {performance['suggested_depth']}")
print_table(performance["candidates"],
            ["depth", "status", "near_seconds", "far_seconds", "evaluation_seconds"],
            {"near_seconds": ".4f", "far_seconds": ".4f", "evaluation_seconds": ".4f"})

`suggest_parameters_for_accuracy` evaluates the exact direct field at a
deterministic sample of targets once, compares every (order, depth) candidate
at those targets and recommends the **fastest** candidate whose sampled RMS
relative error meets the tolerance.

In [ ]:
with quiet_construction():
    accuracy = cdfmm.suggest_parameters_for_accuracy(
        positions, targets, moments, desired_accuracy=1.0e-3,
        candidate_orders=[2, 4, 6], candidate_depths=[1, 2, 3],
        sample_size=128, repetitions=2,
    )
print(f"suggested order {accuracy['suggested_order']}, depth {accuracy['suggested_depth']} "
      f"for RMS relative error <= 1e-3 on {accuracy['reference_target_count']} sampled targets")
print_table(accuracy["candidates"],
            ["order", "depth", "rms_relative_error", "evaluation_seconds", "satisfies_accuracy"],
            {"rms_relative_error": ".2e", "evaluation_seconds": ".4f"})

orders = sorted({c["order"] for c in accuracy["candidates"]})
depths = sorted({c["depth"] for c in accuracy["candidates"]})
grid = np.full((len(depths), len(orders)), np.nan)
for candidate in accuracy["candidates"]:
    grid[depths.index(candidate["depth"]), orders.index(candidate["order"])] = candidate["rms_relative_error"]
figure, axes = plt.subplots(figsize=(5, 3.6))
image = axes.imshow(np.log10(grid), origin="lower", aspect="auto", cmap="viridis")
axes.set_xticks(range(len(orders)), orders)
axes.set_yticks(range(len(depths)), depths)
axes.set(xlabel="expansion order p", ylabel="depth", title="log10 sampled RMS relative error")
if accuracy["suggested_order"] >= 0:
    axes.plot(orders.index(accuracy["suggested_order"]), depths.index(accuracy["suggested_depth"]),
              "wx", markersize=14, markeredgewidth=3)
figure.colorbar(image, ax=axes)
figure.tight_layout()

## Adaptive trees for non-uniform geometry

`AdaptiveTree` splits a box only while it holds more than
`max_particles_per_leaf` bodies, up to `max_depth`, so refined regions get
deep leaves and empty regions none. It produces the same **static topology**
type as the uniform adapter, and `build_fmm` (or `cdfmm.build_static_fmm`)
constructs the ordinary static plan on it. The geometry below is a small
Voronoi-style grain structure refined at the grain boundaries; every cube
contributes one dipole at its centre.

In [ ]:
grains = generate_random_grains(n_grains=4, base_grid=4, n_refine=2, seed=3)
grain_positions = grains["positions"]
grain_moments = grains["moments"]
grain_identities = np.arange(len(grain_positions), dtype=np.int32)
print(f"{len(grain_positions)} dipoles from {grains['material_levels'].max() + 1} refinement levels")

# The grain cubes are dyadic subdivisions of [-0.5, 0.5]^3, so a root box of
# half-width 0.5 would put every dipole on a box boundary; an offset root
# keeps them inside the boxes of both trees.
adaptive_options = cdfmm.AdaptiveTreeOptions()
adaptive_options.max_particles_per_leaf = 16
adaptive_options.max_depth = 4
adaptive_options.root_centre = cdfmm.Vec3(0.013, -0.007, 0.011)
adaptive_options.root_half_width = 0.55
start = time.perf_counter()
adaptive = cdfmm.AdaptiveTree(grain_positions, adaptive_options)
adaptive_seconds = time.perf_counter() - start

uniform_options = cdfmm.UniformTreeOptions()
uniform_options.max_level = adaptive.topology.maximum_level
uniform_options.root_centre = adaptive_options.root_centre
uniform_options.root_half_width = adaptive_options.root_half_width
uniform_topology = cdfmm.uniform_topology(
    cdfmm.UniformTree(grain_positions, grain_positions, uniform_options))

rows = []
for name, topology in (("adaptive", adaptive.topology), ("uniform", uniform_topology)):
    summary = summarise(topology, adaptive_options.max_particles_per_leaf, adaptive_options.max_depth)
    rows.append({"tree": name, "reached depth": summary["reached_depth"], "nodes": summary["nodes"],
                 "occupied leaves": summary["leaves"], "max occupancy": summary["occupancy_max"],
                 "near-field pairs": summary["p2p_pairs"], "M2L": summary["m2l"],
                 "cross-level M2L": summary["cross_level_m2l"]})
print_table(rows)
print(f"adaptive construction {adaptive_seconds * 1e3:.1f} ms "
      f"(tree {adaptive.tree_seconds * 1e3:.1f} ms, interactions {adaptive.interaction_seconds * 1e3:.1f} ms)")

figure = plt.figure(figsize=(12, 5))
for panel, (name, topology) in enumerate((("adaptive", adaptive.topology), ("uniform", uniform_topology)), 1):
    axes = figure.add_subplot(1, 2, panel, projection="3d")
    for leaf in topology.target_leaves:
        node = topology.nodes[leaf.node]
        draw_box_3d(axes, vec3_to_array(node.centre), node.half_width,
                    colour=plt.cm.viridis(node.level / adaptive_options.max_depth), alpha=0.6, linewidth=0.6)
    finish_3d_axes(axes, f"{name}: occupied leaves coloured by level")
figure.tight_layout()

Both topologies feed the same static plan. Cross-level M2L interactions in the
adaptive tree use the same universal operators; the exact near field covers
unequal-size leaf pairs. The two trees also partition differently: the
uniform tree uses the classical `list2` (children of the parent's neighbours),
while the adaptive tree admits a far-field pair only when the enclosing
spheres satisfy $(r_s + r_t)/d \le 0.75$, so it keeps more pairs in the exact
near field and reaches a lower far-field error at the same order and depth.
The persistent geometry cache is disabled for a supplied topology.

In [ ]:
plan_options = cdfmm.UniformFmmOptions()
plan_options.expansion_order = 4
plan_options.precision = cdfmm.StaticPrecision.FLOAT64
plan_options.fixed_target_source_indices = grain_identities.tolist()

sample = np.sort(rng.choice(len(grain_positions), 200, replace=False))
H_direct = cdfmm.direct_p2p_reference(
    grain_positions[sample], grain_positions, grain_moments,
    target_source_indices=sample.tolist(),
)["H"]

rows = []
for name, topology in (("adaptive", adaptive.topology), ("uniform", uniform_topology)):
    with quiet_construction():
        plan = cdfmm.build_static_fmm(topology, plan_options)
    start = time.perf_counter()
    H = plan.evaluate(grain_moments, target_source_indices=grain_identities)["H"]
    seconds = time.perf_counter() - start
    rows.append({"tree": name,
                 "relative L2 (sampled)": field_error_summary(H[sample], H_direct)["relative_l2"],
                 "eval ms": 1e3 * seconds,
                 "host MiB": plan.static_plan_statistics["total_persistent_bytes"] / 2**20})
print_table(rows, formats={"relative L2 (sampled)": ".2e", "eval ms": ".3f", "host MiB": ".2f"})

## Summary

- Depth trades exact near-field pairs for M2L translations; order trades
  accuracy for coefficient count. Neither has a universal optimum.
- Use the advisers once at setup on the real geometry, then fix the returned
  values in the production options.
- `AdaptiveTree` is the tool for strongly non-uniform geometry; both trees are
  consumed by the same plan and executors.
- The `REGULAR_GRID` layout hint (tutorial 3) is the complementary tool for
  perfectly regular lattices.